## summarizing some  reviews before classification

In [ ]:
from transformers import pipeline
import pandas as pd
import torch
import random
import re

In [ ]:
#!pip install torch
# !pip install transformers --upgrade

In [ ]:
## sample data for testing summarization
seed = 3210

all_reviews = pd.read_parquet('/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/collected_reviews_25112025.parquet')
random.seed(seed)
sampled_reviews = all_reviews.dropna(subset=['trans_review']).sample(n=20, random_state=seed).reset_index(drop=True)
sampled_reviews = sampled_reviews[['place_id', 'review_id', 'trans_review']]

In [ ]:
# ------------------------------
# 1. Define only BART and Flan-T5
# ------------------------------
models = {
    "facebook/bart-large-cnn": {"type": "summarization"},
    "google/flan-t5-large": {"type": "text2text-generation"}
}

# ------------------------------
# 2. Load pipelines simply
# ------------------------------
pipelines = {}
for model_name, info in models.items():
    print(f"Loading model: {model_name} ...")
    try:
        pipelines[model_name] = pipeline(
            info["type"],
            model=model_name,
            tokenizer=model_name
        )
    except Exception as e:
        print(f"Failed to load {model_name}: {e}")
        continue

print(f"Successfully loaded {len(pipelines)} models")

In [ ]:
# ------------------------------
# 3. Simple summarization with length filtering
# ------------------------------
summarized_data = []

MIN_CHARS_FOR_SUMMARIZATION = 100
MAX_SUMMARY_CHARS = 200

for r in sampled_reviews.itertuples():
    row = {
        "review_id": r.review_id,
        "place_id": r.place_id,
        "original_text": r.trans_review
    }

    # Clean text
    if pd.isna(r.trans_review) or not isinstance(r.trans_review, str):
        text = ""
    else:
        text = ' '.join(r.trans_review.split())  # Remove extra whitespace
    
    char_count = len(text)
    
    # Skip if text is missing/invalid
    if not text.strip():
        for model_name in pipelines.keys():
            row[model_name] = ""
        summarized_data.append(row)
        continue
    
    # Check if text meets summarization threshold
    should_summarize = char_count > MIN_CHARS_FOR_SUMMARIZATION
    
    for model_name, pipe in pipelines.items():
        if not should_summarize:
            # Use original text for short reviews
            row[model_name] = text
        else:
            try:
                if model_name == "facebook/bart-large-cnn":
                    # Simple BART summarization
                    out = pipe(
                        text,
                        max_length=80,
                        min_length=30,
                        do_sample=False
                    )
                    if out and isinstance(out, list):
                        summary = out[0].get("summary_text", "").strip()
                    else:
                        summary = text
                        
                else:  # Flan-T5
                    # Simple Flan-T5 with basic prompt
                    prompt = f"summarize: {text}"
                    out = pipe(
                        prompt,
                        max_length=80,
                        min_length=30,
                        do_sample=False
                    )
                    if out and isinstance(out, list):
                        summary = out[0].get("generated_text", "").strip()
                        # Remove "summarize: " if present
                        if summary.lower().startswith("summarize:"):
                            summary = summary[10:].strip()
                    else:
                        summary = text
                
                # Ensure summary doesn't exceed character limit
                if len(summary) > MAX_SUMMARY_CHARS:
                    # Truncate at last complete word before limit
                    truncated = summary[:MAX_SUMMARY_CHARS]
                    last_space = truncated.rfind(' ')
                    if last_space > 0:
                        summary = truncated[:last_space] + '...'
                    else:
                        summary = truncated + '...'
                        
            except Exception as e:
                print(f"Error with {model_name} for review {r.review_id}: {e}")
                summary = text  # Fallback to original text
            
            row[model_name] = summary

    # Add metadata
    row["cleaned_text"] = text
    row["char_count"] = char_count
    row["was_summarized"] = should_summarize
    summarized_data.append(row)


# ------------------------------
# 3. Simple summarization with length filtering
# ------------------------------
summarized_data = []

MIN_CHARS_FOR_SUMMARIZATION = 100
MAX_SUMMARY_CHARS = 200

for r in sampled_reviews.itertuples():
    row = {
        "review_id": r.review_id,
        "place_id": r.place_id,
        "original_text": r.trans_review
    }

    # Clean text
    if pd.isna(r.trans_review) or not isinstance(r.trans_review, str):
        text = ""
    else:
        text = ' '.join(r.trans_review.split())  # Remove extra whitespace
    
    char_count = len(text)
    
    # Skip if text is missing/invalid
    if not text.strip():
        for model_name in pipelines.keys():
            row[model_name] = ""
        summarized_data.append(row)
        continue
    
    # Check if text meets summarization threshold
    should_summarize = char_count > MIN_CHARS_FOR_SUMMARIZATION
    
    for model_name, pipe in pipelines.items():
        if not should_summarize:
            # Use original text for short reviews
            row[model_name] = text
        else:
            try:
                if model_name == "facebook/bart-large-cnn":
                    # Simple BART summarization
                    out = pipe(
                        text,
                        max_length=80,
                        min_length=30,
                        do_sample=False
                    )
                    if out and isinstance(out, list):
                        summary = out[0].get("summary_text", "").strip()
                    else:
                        summary = text
                        
                else:  # Flan-T5
                    # Simple Flan-T5 with basic prompt
                    prompt = f"summarize: {text}"
                    out = pipe(
                        prompt,
                        max_length=80,
                        min_length=30,
                        do_sample=False
                    )
                    if out and isinstance(out, list):
                        summary = out[0].get("generated_text", "").strip()
                        # Remove "summarize: " if present
                        if summary.lower().startswith("summarize:"):
                            summary = summary[10:].strip()
                    else:
                        summary = text
                
                # Ensure summary doesn't exceed character limit
                if len(summary) > MAX_SUMMARY_CHARS:
                    # Truncate at last complete word before limit
                    truncated = summary[:MAX_SUMMARY_CHARS]
                    last_space = truncated.rfind(' ')
                    if last_space > 0:
                        summary = truncated[:last_space] + '...'
                    else:
                        summary = truncated + '...'
                        
            except Exception as e:
                print(f"Error with {model_name} for review {r.review_id}: {e}")
                summary = text  # Fallback to original text
            
            row[model_name] = summary

    # Add metadata
    row["cleaned_text"] = text
    row["char_count"] = char_count
    row["was_summarized"] = should_summarize
    summarized_data.append(row)


In [ ]:
# ------------------------------
# 4. Save results
# ------------------------------
df = pd.DataFrame(summarized_data)
csv_file = "/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/sample_20_reviews_bart_flan_simple.csv"
df.to_csv(csv_file, index=False, encoding="utf-8")
print(f"Summaries saved to {csv_file}")

# Print simple statistics
total_reviews = len(summarized_data)
summarized_count = sum(1 for row in summarized_data if row.get("was_summarized", False))

print(f"\nSimple Statistics:")
print(f"Total reviews: {total_reviews}")
print(f"Summarized: {summarized_count}")
print(f"Used original: {total_reviews - summarized_count}")

# Show a couple examples
print(f"\nExample Results:")
summarized_examples = [r for r in summarized_data if r.get("was_summarized")]
for i, example in enumerate(summarized_examples[:2]):
    print(f"\n--- Example {i+1} ---")
    print(f"Original ({example['char_count']} chars): {example['cleaned_text']}")
    print(f"BART: {example.get('facebook/bart-large-cnn', '')}")
    print(f"Flan-T5: {example.get('google/flan-t5-large', '')}")

In [ ]:
import pandas as pd
summaries = pd.read_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/text_reviews_summarized_25112025.parquet")
summaries.info()

In [ ]:
summarized = summaries[['review_id', 'place_id', 'was_summarized', 'summary']]
summarized.info()

In [ ]:
summaries

In [ ]:
summarized.to_parquet("/home/ethel/Documents/RESEARCH/SOCIAL_INTERACTIONS_2025/TheWEBConf_TheWeb4Good_2026/working_data/text_reviews_summarized_25112025.parquet", index=False)